# CNN Regression: CC vs NOCC Comparison -- Pepper Datasets (K-Fold CV)

This notebook trains the same CNN model on **3 Pepper color datasets** (4 experiments total), comparing:
1. **CC** (Color Corrected) images
2. **NOCC** (No Color Correction) images

**Stratified K-Fold Cross-Validation** with automatic fold reduction for small classes.

**Experiments:**
- Comparison_on_Portakabin / CSVS00 (38 images, 9 scores)
- Comparison_on_Portakabin / CSVS01 (38 images, 9 scores)
- Evaluation_on_Greenhouse / CSVS00 (40 images, 6 scores)
- Evaluation_on_Portakabin / CSVS00 (171 images, 9 scores)

## 1. Import Libraries

In [ ]:
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from glob import glob
import copy
from PIL import Image
from tqdm import tqdm
from skimage import io
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import StratifiedKFold, KFold
from scipy.stats import spearmanr
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
mpl.rcParams['figure.figsize'] = 16, 8
plt.rcParams['font.size'] = 10

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Configuration

In [ ]:
# Root folder for all Pepper datasets
PEPPER_ROOT = r'C:\Users\aymen.masmoudi\Desktop\Database_CSVS\CSVS_Pepper_202602_ElEjido'

# The 3 Pepper folders
PEPPER_FOLDERS = [
    'Pepper_COLOR_Comparison_on_Portakabin',
    'Pepper_COLOR_Evaluation_on_Greenhouse',
    'Pepper_COLOR_Evaluation_on_Portakabin',
]

# Image size
IMG_SIZE = 64

# Training parameters
BATCH_SIZE = 32
NUM_EPOCHS = 1000
LEARNING_RATE = 0.001
EARLY_STOPPING_PATIENCE = 30
LR_PATIENCE = 8
LR_FACTOR = 0.5
DEPTH = 16
N_FOLDS = 5

# Composite score weights
# CS = w_r2*R2 + w_spearman*rho + w_mae*(1 - MAE/range) + w_rmse*(1 - RMSE/range)
# Higher CS = better (good metrics maximized, error metrics minimized)
COMPOSITE_WEIGHTS = {'r2': 0.3, 'spearman': 0.3, 'mae': 0.2, 'rmse': 0.2}

print(f"Pepper root: {PEPPER_ROOT}")
print(f"Image size: {IMG_SIZE}x{IMG_SIZE}")
print(f"Batch size: {BATCH_SIZE}, Epochs: {NUM_EPOCHS}, LR: {LEARNING_RATE}")
print(f"LR scheduler: patience={LR_PATIENCE}, factor={LR_FACTOR}")
print(f"Early stopping: {EARLY_STOPPING_PATIENCE}")
print(f"Cross-validation: {N_FOLDS}-fold stratified")
print(f"Composite: {COMPOSITE_WEIGHTS}")

## 3. Data Loading -- Per-Outil (CSVS00, CSVS01 separately)

In [ ]:
def load_outil_data(output_dir, mode, outil):
    # Load images from OUTPUT/<mode>/<outil>/Score=XX/*.tif
    outil_path = os.path.join(output_dir, mode, outil)
    if not os.path.isdir(outil_path):
        return []
    
    data_list = []
    score_dirs = sorted([
        d for d in os.listdir(outil_path)
        if os.path.isdir(os.path.join(outil_path, d)) and d.startswith('Score=')
    ])
    
    for score_folder in score_dirs:
        score = int(score_folder.split('=')[1])
        folder_path = os.path.join(outil_path, score_folder)
        tif_files = glob(os.path.join(folder_path, '*.tif'))
        for img_path in tif_files:
            data_list.append((img_path, float(score)))
        if tif_files:
            print(f"      {score_folder}: {len(tif_files)} images")
    
    return data_list


# Build experiment list: each entry = (name, cc_data, nocc_data, score_range)
experiments = []

for folder_name in PEPPER_FOLDERS:
    folder_path = os.path.join(PEPPER_ROOT, folder_name)
    output_dir = os.path.join(folder_path, 'OUTPUT')
    short_name = folder_name.replace('Pepper_COLOR_', '')
    
    # Detect outil subfolders
    cc_path = os.path.join(output_dir, 'cc')
    if not os.path.isdir(cc_path):
        print(f"  WARNING: {cc_path} not found, skipping")
        continue
    
    outils = sorted([d for d in os.listdir(cc_path) if os.path.isdir(os.path.join(cc_path, d))])
    
    for outil in outils:
        exp_name = f"{short_name}/{outil}"
        print(f"\n{'='*60}")
        print(f"  {exp_name}")
        print(f"{'='*60}")
        
        print(f"    CC:")
        cc_data = load_outil_data(output_dir, 'cc', outil)
        print(f"    -> {len(cc_data)} images")
        
        print(f"    NOCC:")
        nocc_data = load_outil_data(output_dir, 'nocc', outil)
        print(f"    -> {len(nocc_data)} images")
        
        all_scores = sorted(set([s for _, s in cc_data + nocc_data]))
        score_range = max(all_scores) - min(all_scores) if all_scores else 8
        
        experiments.append({
            'name': exp_name,
            'cc_data': cc_data,
            'nocc_data': nocc_data,
            'score_range': score_range,
            'scores': all_scores,
            'output_dir': output_dir,
        })

print(f"\n\nTotal experiments: {len(experiments)}")
for e in experiments:
    print(f"  {e['name']}: CC={len(e['cc_data'])}, NOCC={len(e['nocc_data'])}, scores={[int(s) for s in e['scores']]}")

## 4. Dataset & Model Definition

In [ ]:
class ImageRegressionDataset(Dataset):
    def __init__(self, data_list, img_size, transform=None, target_noise_std=0.0):
        self.data_list = data_list
        self.img_size = img_size
        self.transform = transform
        self.target_noise_std = target_noise_std
    
    def __len__(self):
        return len(self.data_list)
    
    def __getitem__(self, idx):
        img_path, target = self.data_list[idx]
        try:
            img_array = io.imread(img_path)
            if img_array.dtype != np.uint16:
                raise ValueError(f"Image not 16-bit: {img_array.dtype}")
            img_array = img_array.astype(np.float32) / 65535.0
            image = torch.from_numpy(img_array.transpose(2, 0, 1)).float()
            if self.transform:
                image = self.transform(image)
            target = torch.tensor([target], dtype=torch.float32)
            if self.target_noise_std > 0:
                target = target + torch.randn_like(target) * self.target_noise_std
            return image, target
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            image = torch.zeros(3, self.img_size, self.img_size)
            return image, torch.tensor([target], dtype=torch.float32)


class RegressionCNN(nn.Module):
    def __init__(self, input_channels=3, img_height=64, img_width=64, depth=16):
        super(RegressionCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(input_channels, depth, 3, padding=1), nn.BatchNorm2d(depth), nn.ReLU(),
            nn.Conv2d(depth, depth, 3, padding=1), nn.BatchNorm2d(depth), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(depth, depth*2, 3, padding=1), nn.BatchNorm2d(depth*2), nn.ReLU(),
            nn.Conv2d(depth*2, depth*2, 3, padding=1), nn.BatchNorm2d(depth*2), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(depth*2, depth*4, 3, padding=1), nn.BatchNorm2d(depth*4), nn.ReLU(),
            nn.Conv2d(depth*4, depth*4, 3, padding=1), nn.BatchNorm2d(depth*4), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(depth*4, depth*8, 3, padding=1), nn.BatchNorm2d(depth*8), nn.ReLU(),
            nn.Conv2d(depth*8, depth*8, 3, padding=1), nn.BatchNorm2d(depth*8), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(depth*8, depth*16, 3, padding=1), nn.BatchNorm2d(depth*16), nn.ReLU(),
            nn.Conv2d(depth*16, depth*16, 3, padding=1), nn.BatchNorm2d(depth*16), nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(depth*16, depth*32, 3, padding=1), nn.BatchNorm2d(depth*32), nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.flattened_size = depth * img_height * img_width // 128
        self.fc_layers = nn.Sequential(
            nn.Linear(self.flattened_size, self.flattened_size // 2),
            nn.ReLU(),
            nn.Linear(self.flattened_size // 2, self.flattened_size // 4),
            nn.ReLU(),
            nn.Linear(self.flattened_size // 4, 1),
            nn.ELU()
        )
    
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        return x


def get_transforms(img_size):
    # Train and val transforms with enhanced data augmentation
    train_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomChoice([
            transforms.Lambda(lambda x: torch.rot90(x, k=0, dims=[1, 2])),
            transforms.Lambda(lambda x: torch.rot90(x, k=1, dims=[1, 2])),
            transforms.Lambda(lambda x: torch.rot90(x, k=2, dims=[1, 2])),
            transforms.Lambda(lambda x: torch.rot90(x, k=3, dims=[1, 2]))
        ]),
        transforms.RandomAffine(degrees=15, translate=(0.05, 0.05), scale=(0.9, 1.1)),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    val_transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])
    return train_transform, val_transform


def compute_composite_score(r2, spearman, mae, rmse, score_range=8):
    # Composite score: higher is better
    # Maximizes R2, Spearman rho | Minimizes MAE, RMSE
    # CS = 0.3*R2 + 0.3*rho + 0.2*(1 - MAE/range) + 0.2*(1 - RMSE/range)
    w = COMPOSITE_WEIGHTS
    mae_norm = 1.0 - (mae / score_range)
    rmse_norm = 1.0 - (rmse / score_range)
    return w['r2'] * r2 + w['spearman'] * spearman + w['mae'] * mae_norm + w['rmse'] * rmse_norm


print("Classes and functions defined.")
print(f"  Composite: 0.3*R2 + 0.3*rho + 0.2*(1-MAE/range) + 0.2*(1-RMSE/range)")
print(f"  -> Maximizes R2, Spearman | Minimizes MAE, RMSE")

## 5. Training Function (Stratified K-Fold CV)

In [ ]:
def train_model(data_list, img_size, score_range=8, model_name="model", n_folds=N_FOLDS, seed=SEED):
    # Stratified K-Fold CV. Returns fold metrics, OOF predictions, best model.
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    target_bins = np.array([int(t) for _, t in data_list])
    unique_scores, class_counts = np.unique(target_bins, return_counts=True)
    min_count = int(class_counts.min())

    # Auto-reduce folds if any class is too small for stratification
    if min_count >= 2:
        actual_folds = min(n_folds, min_count)
        skf = StratifiedKFold(n_splits=actual_folds, shuffle=True, random_state=seed)
        split_iter = list(skf.split(np.arange(len(data_list)), target_bins))
    else:
        actual_folds = min(n_folds, len(data_list))
        kf = KFold(n_splits=actual_folds, shuffle=True, random_state=seed)
        split_iter = list(kf.split(np.arange(len(data_list))))
        print(f"    WARNING: min class count={min_count}, using regular KFold")

    if actual_folds < n_folds:
        print(f"    Reduced folds {n_folds} -> {actual_folds} (min class count={min_count})")
    print(f"    Running {actual_folds}-fold CV on {len(data_list)} samples ({len(unique_scores)} classes)")

    fold_metrics = []
    fold_histories = []
    oof_preds = np.zeros(len(data_list))
    oof_targets = np.array([t for _, t in data_list])
    best_fold_cs = -float('inf')
    best_model_state = None

    for fold_idx, (train_idx, val_idx) in enumerate(split_iter):
        print(f"\n    --- Fold {fold_idx+1}/{actual_folds} ---")
        train_list = [data_list[i] for i in train_idx]
        val_list = [data_list[i] for i in val_idx]
        print(f"    Train: {len(train_list)} | Val: {len(val_list)}")

        train_tf, val_tf = get_transforms(img_size)
        train_ds = ImageRegressionDataset(train_list, img_size, transform=train_tf, target_noise_std=0.15)
        val_ds = ImageRegressionDataset(val_list, img_size, transform=val_tf)
        train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
        val_dl = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

        model = RegressionCNN(3, img_size, img_size, DEPTH).to(device)
        criterion = nn.MSELoss()
        optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', factor=LR_FACTOR, patience=LR_PATIENCE)

        hist = {'train_loss': [], 'val_loss': [], 'val_mae': [], 'val_rmse': [],
                'val_r2': [], 'val_spearman': [], 'val_composite': [], 'lr': []}
        best_cs = -float('inf')
        best_state = None
        no_improve = 0

        for epoch in range(NUM_EPOCHS):
            # Train
            model.train()
            tl_sum = 0.0
            for imgs, tgts in train_dl:
                imgs, tgts = imgs.to(device), tgts.to(device)
                optimizer.zero_grad()
                out = model(imgs)
                loss = criterion(out, tgts)
                loss.backward()
                optimizer.step()
                tl_sum += loss.item() * imgs.size(0)
            tl = tl_sum / len(train_ds)

            # Validate
            model.eval()
            vl_sum = 0.0
            ep_preds, ep_tgts = [], []
            with torch.no_grad():
                for imgs, tgts in val_dl:
                    imgs, tgts = imgs.to(device), tgts.to(device)
                    out = model(imgs)
                    vl_sum += criterion(out, tgts).item() * imgs.size(0)
                    ep_preds.extend(out.cpu().numpy().flatten())
                    ep_tgts.extend(tgts.cpu().numpy().flatten())

            vl = vl_sum / len(val_ds)
            v_mae = mean_absolute_error(ep_tgts, ep_preds)
            v_rmse = np.sqrt(mean_squared_error(ep_tgts, ep_preds))
            v_r2 = r2_score(ep_tgts, ep_preds)
            sp = spearmanr(ep_tgts, ep_preds).correlation
            if np.isnan(sp):
                sp = 0.0
            v_cs = compute_composite_score(v_r2, sp, v_mae, v_rmse, score_range)
            cur_lr = optimizer.param_groups[0]['lr']
            scheduler.step(vl)

            hist['train_loss'].append(tl)
            hist['val_loss'].append(vl)
            hist['val_mae'].append(v_mae)
            hist['val_rmse'].append(v_rmse)
            hist['val_r2'].append(v_r2)
            hist['val_spearman'].append(sp)
            hist['val_composite'].append(v_cs)
            hist['lr'].append(cur_lr)

            if v_cs > best_cs:
                best_cs = v_cs
                best_state = copy.deepcopy(model.state_dict())
                no_improve = 0
            else:
                no_improve += 1

            if (epoch + 1) % 25 == 0 or epoch == 0:
                print(f"      Ep {epoch+1:4d} | VL:{vl:.4f} R2:{v_r2:.3f} rho:{sp:.3f} CS:{v_cs:.4f}")

            if no_improve >= EARLY_STOPPING_PATIENCE:
                print(f"      Early stop ep {epoch+1} (CS:{best_cs:.4f})")
                break

        # Best model eval for this fold
        model.load_state_dict(best_state)
        model.eval()
        fp, ft = [], []
        with torch.no_grad():
            for imgs, tgts in val_dl:
                out = model(imgs.to(device))
                fp.extend(out.cpu().numpy().flatten())
                ft.extend(tgts.numpy().flatten())
        fp, ft = np.array(fp), np.array(ft)

        oof_preds[val_idx] = fp

        f_r2 = r2_score(ft, fp)
        f_sp = spearmanr(ft, fp).correlation
        if np.isnan(f_sp):
            f_sp = 0.0
        f_mae = mean_absolute_error(ft, fp)
        f_rmse = np.sqrt(mean_squared_error(ft, fp))
        f_cs = compute_composite_score(f_r2, f_sp, f_mae, f_rmse, score_range)

        fm = {'MSE': mean_squared_error(ft, fp), 'RMSE': f_rmse, 'MAE': f_mae,
              'R2': f_r2, 'Spearman': f_sp, 'Composite': f_cs,
              'Epochs': len(hist['train_loss'])}
        fold_metrics.append(fm)
        fold_histories.append(hist)

        print(f"    Fold {fold_idx+1} Final | R2:{f_r2:.4f} rho:{f_sp:.4f} MAE:{f_mae:.3f} CS:{f_cs:.4f}")

        if f_cs > best_fold_cs:
            best_fold_cs = f_cs
            best_model_state = copy.deepcopy(model.state_dict())

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    # Aggregate metrics across folds
    mkeys = ['MSE', 'RMSE', 'MAE', 'R2', 'Spearman', 'Composite', 'Epochs']
    mean_m = {k: np.mean([fm[k] for fm in fold_metrics]) for k in mkeys}
    std_m = {k: np.std([fm[k] for fm in fold_metrics]) for k in mkeys}

    # OOF metrics (all out-of-fold predictions aggregated)
    oof_r2 = r2_score(oof_targets, oof_preds)
    oof_sp = spearmanr(oof_targets, oof_preds).correlation
    if np.isnan(oof_sp):
        oof_sp = 0.0
    oof_mae = mean_absolute_error(oof_targets, oof_preds)
    oof_rmse = np.sqrt(mean_squared_error(oof_targets, oof_preds))
    oof_m = {
        'R2': oof_r2, 'Spearman': oof_sp, 'MAE': oof_mae,
        'RMSE': oof_rmse, 'MSE': mean_squared_error(oof_targets, oof_preds),
        'Composite': compute_composite_score(oof_r2, oof_sp, oof_mae, oof_rmse, score_range),
    }

    print(f"\n    === {model_name} | {actual_folds}-fold CV ===")
    print(f"    Mean  R2:{mean_m['R2']:.4f}+/-{std_m['R2']:.4f}  rho:{mean_m['Spearman']:.4f}+/-{std_m['Spearman']:.4f}")
    print(f"    Mean  MAE:{mean_m['MAE']:.4f}+/-{std_m['MAE']:.4f}  CS:{mean_m['Composite']:.4f}+/-{std_m['Composite']:.4f}")
    print(f"    OOF   R2:{oof_m['R2']:.4f}  rho:{oof_m['Spearman']:.4f}  MAE:{oof_m['MAE']:.4f}  CS:{oof_m['Composite']:.4f}")

    best_model = RegressionCNN(3, img_size, img_size, DEPTH).to(device)
    best_model.load_state_dict(best_model_state)

    return {
        'model': best_model,
        'fold_metrics': fold_metrics,
        'fold_histories': fold_histories,
        'mean_metrics': mean_m,
        'std_metrics': std_m,
        'oof_preds': oof_preds,
        'oof_targets': oof_targets,
        'oof_metrics': oof_m,
        'n_folds': actual_folds,
    }

print(f"train_model() defined -- Stratified {N_FOLDS}-fold CV (auto-reduced for small classes).")

## 6. Train All Experiments (CC & NOCC)

In [ ]:
all_results = {}

for exp in experiments:
    name = exp['name']
    print(f"\n{'='*80}")
    print(f"  EXPERIMENT: {name}")
    print(f"  CC: {len(exp['cc_data'])} | NOCC: {len(exp['nocc_data'])} | Scores: {[int(s) for s in exp['scores']]}")
    print(f"{'='*80}")
    
    all_results[name] = {}
    
    # Train CC
    if len(exp['cc_data']) >= 6:
        print(f"\n  -- CC (Color Corrected) --")
        all_results[name]['CC'] = train_model(
            exp['cc_data'], IMG_SIZE,
            score_range=exp['score_range'],
            model_name=f"CC_{name}"
        )
    else:
        print(f"  Skipping CC: only {len(exp['cc_data'])} images")
    
    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    
    # Train NOCC
    if len(exp['nocc_data']) >= 6:
        print(f"\n  -- NOCC (No Color Correction) --")
        all_results[name]['NOCC'] = train_model(
            exp['nocc_data'], IMG_SIZE,
            score_range=exp['score_range'],
            model_name=f"NOCC_{name}"
        )
    else:
        print(f"  Skipping NOCC: only {len(exp['nocc_data'])} images")
    
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

print(f"\n{'='*80}")
print("ALL TRAINING COMPLETE!")
print(f"{'='*80}")

## 7. Results Summary

In [ ]:
rows = []
for exp in experiments:
    name = exp['name']
    if name not in all_results:
        continue
    for mode in ['CC', 'NOCC']:
        if mode not in all_results[name]:
            continue
        res = all_results[name][mode]
        mm = res['mean_metrics']
        sm = res['std_metrics']
        om = res['oof_metrics']
        data_len = len(exp['cc_data']) if mode == 'CC' else len(exp['nocc_data'])
        rows.append({
            'Experiment': name,
            'Mode': mode,
            'N': data_len,
            'Folds': res['n_folds'],
            'OOF_R2': f"{om['R2']:.4f}",
            'OOF_rho': f"{om['Spearman']:.4f}",
            'OOF_MAE': f"{om['MAE']:.4f}",
            'OOF_RMSE': f"{om['RMSE']:.4f}",
            'OOF_CS': f"{om['Composite']:.4f}",
            'Mean_CS+/-Std': f"{mm['Composite']:.3f}+/-{sm['Composite']:.3f}",
            'Epochs': f"{mm['Epochs']:.0f}",
        })

results_df = pd.DataFrame(rows)
print("=" * 140)
print("FULL RESULTS: Pepper CC vs NOCC -- Stratified K-Fold CV")
print("=" * 140)
print(results_df.to_string(index=False))
print("=" * 140)

# Per-experiment winner (by OOF Composite)
print("\nPer-experiment winner (by OOF Composite Score):")
for exp in experiments:
    name = exp['name']
    if name not in all_results:
        continue
    modes = all_results[name]
    if 'CC' in modes and 'NOCC' in modes:
        cc_cs = modes['CC']['oof_metrics']['Composite']
        nocc_cs = modes['NOCC']['oof_metrics']['Composite']
        winner = 'CC' if cc_cs > nocc_cs else 'NOCC'
        delta = abs(cc_cs - nocc_cs)
        print(f"  {name}: {winner} wins (CC={cc_cs:.4f}, NOCC={nocc_cs:.4f}, delta={delta:.4f})")

## 8. Training Curves

In [ ]:
active_exps = [e for e in experiments if e['name'] in all_results]
n_exps = len(active_exps)

fig, axes = plt.subplots(n_exps, 4, figsize=(22, 5*n_exps))
if n_exps == 1:
    axes = axes.reshape(1, -1)

for row, exp in enumerate(active_exps):
    name = exp['name']
    modes = all_results[name]

    for mode, color in [('CC', 'blue'), ('NOCC', 'red')]:
        if mode not in modes:
            continue
        histories = modes[mode]['fold_histories']
        n_f = len(histories)

        for fi, h in enumerate(histories):
            ep = range(1, len(h['train_loss'])+1)
            alpha = 0.35 if n_f > 1 else 1.0
            lw = 1.0 if n_f > 1 else 2.0
            lbl = f"{mode} ({n_f}f)" if fi == 0 else None

            axes[row, 0].plot(ep, h['val_loss'], color=color, lw=lw, alpha=alpha, label=lbl)
            axes[row, 1].plot(ep, h['val_r2'], color=color, lw=lw, alpha=alpha, label=lbl)
            axes[row, 2].plot(ep, h['val_spearman'], color=color, lw=lw, alpha=alpha, label=lbl)
            axes[row, 3].plot(ep, h['val_composite'], color=color, lw=lw, alpha=alpha, label=lbl)

    titles = ['Val Loss (lower)', 'Val R2 (higher)', 'Val Spearman (higher)', 'Composite (higher)']
    ylabels = ['Loss', 'R2', 'Spearman', 'CS']
    for col in range(4):
        axes[row, col].set_title(f'{name}\n{titles[col]}', fontweight='bold', fontsize=9)
        axes[row, col].set_xlabel('Epoch')
        axes[row, col].set_ylabel(ylabels[col])
        axes[row, col].legend()
        axes[row, col].grid(True, alpha=0.3)

plt.suptitle('Training Curves: CC vs NOCC -- Pepper (K-Fold CV)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Predictions vs True (Scatter)

In [ ]:
fig, axes = plt.subplots(n_exps, 2, figsize=(14, 6*n_exps))
if n_exps == 1:
    axes = axes.reshape(1, -1)

for row, exp in enumerate(active_exps):
    name = exp['name']
    modes = all_results[name]

    for col, (mode, color) in enumerate([('CC', 'steelblue'), ('NOCC', 'salmon')]):
        if mode not in modes:
            axes[row, col].text(0.5, 0.5, f'{mode} N/A', ha='center', va='center', fontsize=14)
            axes[row, col].set_title(f'{mode} -- {name}')
            continue

        res = modes[mode]
        preds = res['oof_preds']
        targets = res['oof_targets']
        om = res['oof_metrics']
        scores = exp['scores']
        nf = res['n_folds']

        axes[row, col].scatter(targets, preds, alpha=0.7, s=80, c=color, edgecolors='black', linewidth=0.5)
        axes[row, col].plot([min(scores), max(scores)], [min(scores), max(scores)], 'k--', lw=2, label='Perfect')
        axes[row, col].set_xlabel('True Score', fontweight='bold')
        axes[row, col].set_ylabel('Predicted Score', fontweight='bold')
        axes[row, col].set_title(
            f'{mode} -- {name} ({nf}-fold OOF)\n'
            f'R2={om["R2"]:.4f} | rho={om["Spearman"]:.4f} | MAE={om["MAE"]:.3f} | CS={om["Composite"]:.4f}',
            fontweight='bold', fontsize=9
        )
        axes[row, col].legend()
        axes[row, col].grid(True, alpha=0.3)

plt.suptitle('OOF Predictions -- Pepper Datasets (K-Fold CV)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Metrics Bar Charts

In [ ]:
exp_names = [e['name'].replace('_on_', '\n') for e in active_exps]
x = np.arange(len(active_exps))
width = 0.35

fig, axes = plt.subplots(2, 3, figsize=(22, 10))
metrics_plot = [
    ('MAE', 'lower is better'),
    ('RMSE', 'lower is better'),
    ('MSE', 'lower is better'),
    ('R2', 'higher is better'),
    ('Spearman', 'higher is better'),
    ('Composite', 'higher is better'),
]

for ax, (metric, direction) in zip(axes.flatten(), metrics_plot):
    cc_vals, nocc_vals = [], []
    cc_errs, nocc_errs = [], []
    for e in active_exps:
        name = e['name']
        if 'CC' in all_results[name]:
            cc_vals.append(all_results[name]['CC']['mean_metrics'][metric])
            cc_errs.append(all_results[name]['CC']['std_metrics'][metric])
        else:
            cc_vals.append(0); cc_errs.append(0)
        if 'NOCC' in all_results[name]:
            nocc_vals.append(all_results[name]['NOCC']['mean_metrics'][metric])
            nocc_errs.append(all_results[name]['NOCC']['std_metrics'][metric])
        else:
            nocc_vals.append(0); nocc_errs.append(0)

    bars1 = ax.bar(x - width/2, cc_vals, width, yerr=cc_errs, capsize=4,
                   label='CC', color='steelblue', edgecolor='black')
    bars2 = ax.bar(x + width/2, nocc_vals, width, yerr=nocc_errs, capsize=4,
                   label='NOCC', color='salmon', edgecolor='black')

    ax.set_xlabel('Experiment')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} ({direction}) -- mean +/- std', fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(exp_names, fontsize=7)
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')

    for bar in list(bars1) + list(bars2):
        h = bar.get_height()
        if h != 0:
            ax.annotate(f'{h:.3f}', xy=(bar.get_x() + bar.get_width()/2, h),
                        xytext=(0, 3), textcoords='offset points', ha='center', fontsize=7)

plt.suptitle('CC vs NOCC Metrics -- Pepper (K-Fold CV, mean +/- std)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 11. Save Results

In [ ]:
# Save summary CSV
results_df.to_csv(os.path.join(PEPPER_ROOT, 'pepper_cc_vs_nocc_results_cv.csv'), index=False)
print(f"Summary CSV: {os.path.join(PEPPER_ROOT, 'pepper_cc_vs_nocc_results_cv.csv')}")

# Save per-experiment details
for exp in active_exps:
    name = exp['name']
    output_dir = os.path.join(exp['output_dir'], 'trainings', name.replace('/', '_') + '_cv')
    os.makedirs(output_dir, exist_ok=True)

    for mode in ['CC', 'NOCC']:
        if mode not in all_results[name]:
            continue

        prefix = mode.lower()
        res = all_results[name][mode]

        # OOF predictions
        pd.DataFrame({
            'true': res['oof_targets'],
            'predicted': res['oof_preds']
        }).to_csv(os.path.join(output_dir, f'{prefix}_oof_predictions.csv'), index=False)

        # Per-fold metrics
        pd.DataFrame(res['fold_metrics']).to_csv(
            os.path.join(output_dir, f'{prefix}_fold_metrics.csv'), index=False)

        # Per-fold training histories
        for fi, hist in enumerate(res['fold_histories']):
            pd.DataFrame(hist).to_csv(
                os.path.join(output_dir, f'{prefix}_history_fold{fi+1}.csv'), index=False)

        # Best model weights
        torch.save(res['model'].state_dict(),
                   os.path.join(output_dir, f'{prefix}_best_model.pth'))

    print(f"  {name} -> {output_dir}")

print("\nAll results saved!")

In [ ]:
# Per-class summary statistics using OOF predictions
print("=" * 80)
print("PER-CLASS SUMMARY STATISTICS (Out-of-Fold Predictions)")
print("=" * 80)

for exp in active_exps:
    name = exp['name']
    for mode in ['CC', 'NOCC']:
        if mode not in all_results[name]:
            continue
        res = all_results[name][mode]
        preds = res['oof_preds']
        targets = res['oof_targets']

        results_df_exp = pd.DataFrame({
            'true_value': targets,
            'predicted_value': preds,
            'absolute_error': np.abs(preds - targets),
            'squared_error': (preds - targets) ** 2
        })

        print(f"\n{'='*60}")
        print(f"  {name} | {mode} ({res['n_folds']}-fold OOF)")
        print(f"{'='*60}")

        for target_cls in sorted(results_df_exp['true_value'].unique()):
            subset = results_df_exp[results_df_exp['true_value'] == target_cls]
            mean_pred = subset['predicted_value'].mean()
            std_pred = subset['predicted_value'].std()
            mean_ae = subset['absolute_error'].mean()
            print(f"  Class {int(target_cls):2d}: n={len(subset):3d} | "
                  f"mean_pred={mean_pred:.3f} +/- {std_pred:.3f} | MAE={mean_ae:.3f}")

        om = res['oof_metrics']
        mse = np.mean(results_df_exp['squared_error'])
        rmse = np.sqrt(mse)
        print(f"\n  OVERALL: MSE={mse:.4f}  RMSE={rmse:.4f}  "
              f"MAE={om['MAE']:.4f}  R2={om['R2']:.4f}  "
              f"rho={om['Spearman']:.4f}  CS={om['Composite']:.4f}")

        # Save per-experiment CSV
        out_dir = os.path.join(exp['output_dir'], 'trainings',
                               name.replace('/', '_') + '_cv')
        os.makedirs(out_dir, exist_ok=True)
        results_df_exp.to_csv(
            os.path.join(out_dir, f'{mode.lower()}_oof_class_results.csv'),
            index=False)

print("\n" + "=" * 80)
print("Per-class results saved.")

In [ ]:
# Predict/collect all images for frieze visualization
print("Loading images for frieze visualization...")

frieze_data = {}
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

for exp in active_exps:
    name = exp['name']
    frieze_data[name] = {}
    for mode in ['CC', 'NOCC']:
        if mode not in all_results[name]:
            continue
        data = exp['cc_data'] if mode == 'CC' else exp['nocc_data']
        imgs = []
        for img_path, _ in data:
            img_array = io.imread(img_path).astype(np.float32) / 65535.0
            t = torch.from_numpy(img_array.transpose(2, 0, 1)).float()
            t = val_tf(t)
            imgs.append(t.permute(1, 2, 0).numpy())  # Convert to (H, W, C)
        frieze_data[name][mode] = np.array(imgs)
        print(f"  {name} | {mode}: {len(imgs)} images loaded ({IMG_SIZE}x{IMG_SIZE})")

print("All images loaded for frieze visualization.")

## 12. Draw frieze of predictions

In [ ]:
# Frieze of images arranged according to OOF predictions (floor rounding)
for exp in active_exps:
    name = exp['name']
    max_score = int(max(exp['scores']))
    for mode in ['CC', 'NOCC']:
        if mode not in all_results[name]:
            continue
        all_images = frieze_data[name][mode]
        all_preds = all_results[name][mode]['oof_preds']

        sorted_indices = np.argsort(all_preds)
        sorted_images = all_images[sorted_indices]
        sorted_preds = all_preds[sorted_indices]

        n = len(sorted_images)
        fig, ax = plt.subplots(1, n + max_score, figsize=(n + max_score, 1))
        class_int = 0
        for i, (img, pred) in enumerate(zip(sorted_images, sorted_preds)):
            img = img * 0.5 + 0.5
            img = np.clip(img, 0, 1)
            class_int_tmp = int(np.floor(pred))
            if class_int_tmp > class_int:
                ax[i+class_int].axis('off')
                ax[i+class_int].text(0.5, 0.5, f'{class_int_tmp}', fontsize=30,
                                     fontweight='bold', ha='center', va='center')
                while class_int < class_int_tmp:
                    class_int += 1
                    ax[i+class_int].set_title(f'{pred:.2f}', fontsize=12)
                    ax[i+class_int].imshow(img)
                    ax[i+class_int].axis('off')
            else:
                ax[i+class_int].set_title(f'{pred:.2f}', fontsize=12)
                ax[i+class_int].imshow(img)
                ax[i+class_int].axis('off')
        while class_int < max_score:
            class_int += 1
            if n + class_int - 1 < len(ax):
                ax[n+class_int-1].axis('off')
                ax[n+class_int-1].text(0.5, 0.5, f'{class_int}', fontsize=30,
                                       fontweight='bold', ha='center', va='center')
        plt.suptitle(f'Frieze by Predictions (floor) -- {name} {mode}',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

In [ ]:
# Frieze of images arranged according to OOF predictions (round rounding)
for exp in active_exps:
    name = exp['name']
    max_score = int(max(exp['scores']))
    for mode in ['CC', 'NOCC']:
        if mode not in all_results[name]:
            continue
        all_images = frieze_data[name][mode]
        all_preds = all_results[name][mode]['oof_preds']

        sorted_indices = np.argsort(all_preds)
        sorted_images = all_images[sorted_indices]
        sorted_preds = all_preds[sorted_indices]

        n = len(sorted_images)
        fig, ax = plt.subplots(1, n + max_score, figsize=(n + max_score, 1))
        class_int = 0
        for i, (img, pred) in enumerate(zip(sorted_images, sorted_preds)):
            img = img * 0.5 + 0.5
            img = np.clip(img, 0, 1)
            class_int_tmp = int(np.round(pred))
            if class_int_tmp > class_int:
                ax[i+class_int].axis('off')
                ax[i+class_int].text(0.5, 0.5, f'{class_int_tmp}', fontsize=30,
                                     fontweight='bold', ha='center', va='center')
                while class_int < class_int_tmp:
                    class_int += 1
                    ax[i+class_int].set_title(f'{pred:.2f}', fontsize=12)
                    ax[i+class_int].imshow(img)
                    ax[i+class_int].axis('off')
            else:
                ax[i+class_int].set_title(f'{pred:.2f}', fontsize=12)
                ax[i+class_int].imshow(img)
                ax[i+class_int].axis('off')
        while class_int < max_score:
            class_int += 1
            if n + class_int - 1 < len(ax):
                ax[n+class_int-1].axis('off')
                ax[n+class_int-1].text(0.5, 0.5, f'{class_int}', fontsize=30,
                                       fontweight='bold', ha='center', va='center')
        plt.suptitle(f'Frieze by Predictions (round) -- {name} {mode}',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

## 13. Draw frieze of annotations

In [ ]:
# Frieze of images arranged according to true annotations
for exp in active_exps:
    name = exp['name']
    max_score = int(max(exp['scores']))
    for mode in ['CC', 'NOCC']:
        if mode not in all_results[name]:
            continue
        all_images = frieze_data[name][mode]
        all_true = all_results[name][mode]['oof_targets']

        sorted_indices = np.argsort(all_true)
        sorted_images = all_images[sorted_indices]
        sorted_true = all_true[sorted_indices]

        n = len(sorted_images)
        fig, ax = plt.subplots(1, n + max_score, figsize=(n + max_score, 1))
        class_int = 0
        for i, (img, true_val) in enumerate(zip(sorted_images, sorted_true)):
            img = img * 0.5 + 0.5
            img = np.clip(img, 0, 1)
            class_int_tmp = int(np.floor(true_val))
            if class_int_tmp > class_int:
                ax[i+class_int].axis('off')
                ax[i+class_int].text(0.5, 0.5, f'{class_int_tmp}', fontsize=30,
                                     fontweight='bold', ha='center', va='center')
                while class_int < class_int_tmp:
                    class_int += 1
                    ax[i+class_int].set_title(f'{true_val:.2f}', fontsize=12)
                    ax[i+class_int].imshow(img)
                    ax[i+class_int].axis('off')
            else:
                ax[i+class_int].set_title(f'{true_val:.2f}', fontsize=12)
                ax[i+class_int].imshow(img)
                ax[i+class_int].axis('off')
        while class_int < max_score:
            class_int += 1
            if n + class_int - 1 < len(ax):
                ax[n+class_int-1].axis('off')
                ax[n+class_int-1].text(0.5, 0.5, f'{class_int}', fontsize=30,
                                       fontweight='bold', ha='center', va='center')
        plt.suptitle(f'Frieze by Annotations -- {name} {mode}',
                     fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()